# Import Statements

In [47]:
import pandas as pd
df = pd.read_csv("2026dependenciesHWFile.csv")
df.head()

,course,courseLetter,courseNumber,relatedTo,relatedToCourseLetter,relatedToCourseNumber,How,Unnamed: 7,courseDictionary,courseLetter.1,courseNumber.1
0,AME 105,AME,105,MUH 105,MUH,105,CrossList,NaN,AME 105,AME,105
1,AME 105,AME,105,Cultural Perspectives,NaN,NaN,GenEd,NaN,AME 121,AME,121
2,AME 105,AME,105,Fine Arts,NaN,NaN,GenEd,NaN,AME 122,AME,122
3,AME 105,AME,105,Humanities,NaN,NaN,GenEd,NaN,AME 205,AME,205
4,AME 121,AME,121,American Studies Minor,NaN,NaN,Checksheet,NaN,AME 264,AME,264


### Computing Courses

**Question 1**\
How many CIS courses at the 100 level are part of the Computer Information Systems, BS program?


In [50]:
df['courseNumber'] = df['courseNumber'].apply(str)

cis100_bs_df = df[
    (df['courseLetter'] == 'CIS') &
    (df['relatedTo'] == 'Computer Information Systems, BS') &
    (df['courseNumber'].str.startswith('1'))
    ]
cis100_bs_df.shape[0]

5

**Question 2**\
How many courses are offered with the CIS designation?

In [51]:
# cis_total_df = df[df.iloc[:, 9] == 'CIS']
cis_total_df = df[df.iloc[:, 9] == 'CIS']
cis_total_df.shape[0]

76

**Question 3**\
How many courses are offered with the CYB designation?

In [52]:
cyb_total_df = df[df.iloc[:, 9] == 'CYB']
cyb_total_df.shape[0]

40

**Question 4**\
How many courses are offered with the DSC designation?

In [53]:
dsc_total_df = df[df.iloc[:, 9] == 'DSC']
dsc_total_df.shape[0]

23

**Question 5**\
How many courses are offered with the ISS designation?

In [54]:
iss_total_df = df[df.iloc[:, 9] == 'ISS']
iss_total_df.shape[0]

54

### Crosslisting of Courses

**Create a dataframe with course designated as "CrossList"**

In [55]:
cl_df = df[df['How'] == 'CrossList']
cl_df.shape[0]

376

**Question 6**\
Find how many cross lists each course has.  How may courses are crosslisted with POS 223?

In [56]:
pos_223_cl_total = (cl_df['relatedTo'] == 'POS 223').sum()
print(pos_223_cl_total)

2


**Question 7**\
What is the mean number of crosslists per course?

In [84]:
cl_total = len(cl_df)
# courses_total = df['course'].nunique()
# print(courses_total)
courses_total = df['courseDictionary'].count()
mean_cl_per_course = cl_total / courses_total
print(round(mean_cl_per_course, 2))

0.28


**Question 8**\
What is the median number of courses crosslisted?

In [65]:
non_cl_df = df[df['How'] != 'CrossList']
non_cl_series = pd.Series(0, index=non_cl_df.index)
cl_series = pd.Series(1, index=cl_df.index)

all_courses = pd.concat([non_cl_series, cl_series])
median = all_courses.median()

print(median)

0.0


**Question 9**\
What is the mode of the number of crosslisted courses?

In [66]:
mode = all_courses.mode().values[0]
print(mode)

0


**Question 10**\
Given that a course is actually crosslisted, what is the average number of courses it is crosslisted to?

In [67]:
course_cl = cl_df.groupby('course')['relatedTo'].count()
cl_mean = course_cl.mean()
print(round(cl_mean, 2))

1.39


**Question 11**\
How many courses have four designations (crosslisted designations count as 1 course in total)?  Note:  what would that look like in this dataset?

In [69]:
courses_with_four_designations = course_cl.value_counts()[3] / 4
print(int(courses_with_four_designations))

5


**Question 12**\
Assuming all of the crosslists are one course (e.g. CIS 255 and DSC 255 are one course), how many courses does UMA offer in its catalog?

In [70]:
two_designated_courses = course_cl.value_counts()[1]
three_designated_courses = course_cl.value_counts()[2]
four_designated_courses = course_cl.value_counts()[3]

total_cl_courses = two_designated_courses + three_designated_courses + four_designated_courses
total_courses_with_designations = (two_designated_courses / 2) + (three_designated_courses / 3) + (four_designated_courses / 4)

total_UMA_course_offerings = df["courseDictionary"].count() - (total_cl_courses - total_courses_with_designations)

print(int(total_UMA_course_offerings))

1214


**Question 13**\
All four of the designations CIS, CYB, DSC, and ISS are offered by the computing group.  How many specific courses are offered by the computing group.  Note:  courses crosslisted together are ONE course for this purpose.

In [100]:
comp_group_df = df[df['courseLetter'].isin(['CIS', 'CYB', 'DSC', 'ISS'])]
comp_group_cl_df = comp_group_df[comp_group_df['How'] == 'CrossList']
# all_comp_cl_df.shape[0]
total_comp_cl = comp_group_cl_df.groupby('course')['relatedTo'].count()
print(total_comp_cl.value_counts())
# with pd.option_context('display.max_rows', None, 'display.max_columns', None):  # more options can be specified also
#     print(comp_group_df)

relatedTo
1    18
2    10
3     2
Name: count, dtype: int64


### Architecture Program
In this section, we will be studying the courses and programs offered by the Architecture faculty.

**Question 14**\
What proportion of courses (by code – crosslists are separate for this purpose) are in the Architecture, B.Arch checksheet?

In [87]:
arch_checksheet_df = df[
    (df['relatedTo'] == 'Architecture, B.Arch') &
    (df['How'] == 'Checksheet')
    ]

prop_barch = len(arch_checksheet_df) / courses_total
print(round(prop_barch, 3))

0.035


**Question 15**\
What proportion of courses (by code) are ARC courses?

In [89]:
arc_total_df = df[df.iloc[:, 9] == 'ARC']
prop_arc = len(arc_total_df) / courses_total
print(round(prop_arc, 3))

0.028


**Question 16**\
Assuming these are independent, what proportion of courses (by code) are both in the Architecture, B.Arch checksheet and are ARC courses?

In [99]:
barch_arc_df = df[
    (df['relatedTo'] == 'Architecture, B.Arch') &
    (df['courseLetter'] == 'ARC')
    ]
prop_barch_arc = len(barch_arc_df) / courses_total
print(round(prop_barch_arc, 3))

0.021


**Question 17**\
What is the conditional probability that a given course with an ARC designation is part of the Architecture, B.Arch checksheet?